# ST 554 Final Project

*By: Cass Crews*

## Introduction


## Module Loading and Data Cleaning

Before we load the data and clean it, we need to load the modules and functions we will need in this notebook. 

In [1]:
# Reading in modules and sub-modules, also initiating spark sequence
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, \
    VectorIndexer, OneHotEncoder, Interaction, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import PCA
from pyspark.sql.types import StructType
from pyspark.sql.functions import col
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 12:30:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


We are now ready to load the dataset. We'll initially read it in as a `pandas` dataframe before converting to a SQL-style `pyspark` dataframe. Once we've read the file in, we will extract the first few rows to ensure the data loaded correctly. 

In [2]:
# Reading in the data
power_pddf = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

# Printing the first few rows
power_pddf.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


This looks correct. Let's apply a few validation checks before converting to a `pyspark` dataframe. We'll start by checking for missing values. 

In [3]:
# Checking for missing values
power_pddf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47174 entries, 0 to 47173
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Temperature            47174 non-null  float64
 1   Humidity               47174 non-null  float64
 2   Wind_Speed             47174 non-null  float64
 3   General_Diffuse_Flows  47174 non-null  float64
 4   Diffuse_Flows          47174 non-null  float64
 5   Power_Zone_1           47174 non-null  float64
 6   Power_Zone_2           47174 non-null  float64
 7   Power_Zone_3           47174 non-null  float64
 8   Month                  47174 non-null  int64  
 9   Hour                   47174 non-null  int64  
dtypes: float64(8), int64(2)
memory usage: 3.6 MB


None of the variables are missing, and all are stored as numeric -- as they should be. 

Of course, there could be hidden missing values that have a non-`NaN` missing value code. We can generate some basic summary statistics in an attempt to surface such values. 

In [4]:
# Generating validation summary statistics
power_pddf.describe()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
count,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000,47174.000000
mean,18.813220,68.288398,1.961621,182.531180,74.987211,32335.168690,21027.204976,17831.197608,6.510599,11.488383
std,5.813341,15.560330,2.349351,264.431856,124.256146,7130.013305,5199.787153,6622.590470,3.437367,6.921920
min,3.247000,11.340000,0.050000,0.004000,0.011000,13895.696200,8560.081466,5935.174070,1.000000,0.000000
25%,14.420000,58.320000,0.078000,0.062000,0.122000,26289.581305,16956.121510,13121.927710,4.000000,5.000000
50%,18.780000,69.890000,0.086000,4.829000,4.279500,32261.596960,20804.671935,16406.308170,7.000000,11.000000
75%,22.910000,81.500000,4.915000,318.900000,101.000000,37317.446810,24697.888273,21633.372830,9.000000,17.000000
max,40.010000,94.800000,6.483000,1163.000000,936.000000,52146.859050,37408.860760,47598.326360,12.000000,23.000000


In [5]:
power_pddf.Month.value_counts()

Month
3     4057
7     4029
10    4026
1     4014
8     3999
5     3997
6     3913
9     3913
4     3893
11    3877
12    3868
2     3588
Name: count, dtype: int64

While I am not an expert in any of these metrics, none of the minimum or maximum values clearly indicate missing value codes. The dataset's UCI Machine Learning Repository page indicated no true missing values, but it's always good practice to check for ourselves! 

We are ready to convert to a `pyspark` dataframe. After the conversion, we will again extract the first few rows to confirm they match the first few rows of the `pandas` dataframe. 

In [6]:
# Converting to a pyspark dataframe
power_df = spark.createDataFrame(power_pddf)

# Printing first five rows
power_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

Everything seems to be in order. Onto building the machine learning model! 

## Fitting the Model

As was mentioned in the introduction, we will tune an elastic net model for the purpose of predicting `Power_Zone_3`. We will construct an `MLlib` pipeline to do so, as this will make it much easier to predict power consumption for streamed values. Thus, we need to construct the appropriate transformers. 

We'll work through these sequentially, starting with the creation of a binary predictor from `Hour`, the hour of the day. First, we need to determine how `Hour` was stored when we converted to the `pyspark` dataframe. 

In [7]:
power_df.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

`Hour` is stored as a `bigint`, so we will need to cast it to a `double` type before we create the binary variable; this binary variable will indicate whether the observation corresponds to the early-morning hours (between midnight and 6am) or not, so we need to be able to pass `Binarizer` a threshold that is not an integer. 

In [8]:
# Creating double version of Hour
hour_conv_step = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_db FROM __THIS__
    """)

We can use this `double`-cast column to generate the early-morning binary variable. 

In [9]:
# Creating binary early morning indicator
hour_binar_step = Binarizer(threshold = 6.5, inputCol = "Hour_db", outputCol = "early_morn_bin")

Our next transformer should create month dummies from `Month`. We'll use a one-hot encoder transformer for this step. However, 

In [10]:
# Creating month dummies
month_dummy_step = OneHotEncoder(inputCol = "Month", outputCol = "Month_enc")

Note that the output column is a single column although we created dummies for each month. This is because `OneHotEncoder()` expects the dummies to be added to a single features column of downstream, and, therefore, stores an observation's month dummy values in a vector rather than creating multiple unnecessary columns. 

The next step of the pipeline is more complex. We want to distill the atmospheric variables into their principle components. Thus, we will fit a *Principal Components Analysis (PCA)* to `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`. We will then retain the two principal components that account for the most variability across the atmospheric variables. 

Our first step is to combine the atmospheric variables into a single `features_pca_raw` column to be fed to the `PCA()` transformer. 

In [11]:
# Creating feature column for PCA
pca_assembly_step = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
                              outputCol = "features_pca_raw")

We now need to standardize each feature, as different scales can influence a PCA fit. This simply requires us to pass in the `features_pca_raw` column, as `StandardScaler()` will parse the column and standardize each feature. 

In [12]:
# Standardizing features for PCA
pca_scaling_step = StandardScaler(inputCol = "features_pca_raw", outputCol = "features_pca_stand", withMean = True, withStd = True)

We can now apply PCA without worrying about any scaling issues. In specifying `k = 2`, this transformer will return the first two principal components in a single column of vectors; each vector will be of length 2. 

In [13]:
# Applying PCA to scaled atmospheric features
pca_step = PCA(inputCol = "features_pca_stand", outputCol = "princ_comps", k = 2)

As we are constructing a pipeline that leads to an elastic net model, we need to rename `Power_Zone_3` `label` in line with MLlib convention. We'll use this as an opportunity to remove the variables we no longer need. 

In [14]:
# Renaming Power_Zone_3 label and dropping some variables
clean_step = SQLTransformer(statement = """
    SELECT Power_Zone_3 AS label, princ_comps, early_morn_bin, Power_Zone_1, Power_Zone_2, Month_enc
    FROM __THIS__
    """)

We are ready for the final transformation step: to compile the principal components, `early_morn_bin`, `Power_Zone_1`, `Power_Zone_2`, and `Month_enc` into a single `features` column.

In [15]:
# Compiling final feature set into a features column
assembly_step = VectorAssembler(inputCols = ["princ_comps", "early_morn_bin", "Power_Zone_1", "Power_Zone_2", "Month_enc"], 
                                outputCol = "features")

It's time to specify the model for our pipeline, construct the pipeline, and construct a grid of hyperparameters. As the elastic net model involves both an $L_1$ and an $L_2$ penalty, we will naturally need to tune two hyperparameters. When using the `LinearRegression()` estimator to fit an elastic net model, the two hyperparameters are `regParam` and `elasticNetParam`. `regParam` determines the overall regularization amount across both penalties, while `elasticNetParam` determines the relative severity of each penalty; `elasticNetParam = 0` corresponds to a pure ridge regression, and `elasticNetParam = 1` corresponds to a pure LASSO regression. 

For each hyperparameter, we will consider the same range of 11 values: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, and 1. We will evaluate predictive performance for each combination of hyperparameter values, an 11-by-11 grid of 121 combinations. 

An important note: `LinearRegression()` handles feature standardization for us via the default `standardization=True`, meaning we do not need to add another standardization transformation to the pipeline prior to fitting elastic net models. 

In [16]:
# Specifying the estimator
en_model = LinearRegression()

# Constructing the pipeline
pipeline = Pipeline(stages = [hour_conv_step, hour_binar_step, month_dummy_step, pca_assembly_step, pca_scaling_step, pca_step, clean_step, assembly_step, en_model])

# Constructing the 11x11 hyperparameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(en_model.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(en_model.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

We are now ready to construct a `CrossValidator()` object, specifying that we want to use 5-fold cross validations to tune the elastic net model and select the best overall model based on cross validated root mean squared error (RMSE). Once we have constructed the object, we will fit it to identify the best combination of hyperparameters. 

In [17]:
# Constructing CrossValidator object
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(), # rmse by default
                          numFolds = 5, # 5-fold CV
                          seed = 5) # setting seed for reproducibility

# Fitting the object
en_cv = crossval.fit(power_df)

26/04/26 12:30:34 WARN Instrumentation: [32da0f62] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:37 WARN Instrumentation: [4e2fe144] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:39 WARN Instrumentation: [b1d6108f] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:40 WARN Instrumentation: [ff009207] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:42 WARN Instrumentation: [8fabeb30] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:43 WARN Instrumentation: [3e9af59f] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:45 WARN Instrumentation: [90fa3b27] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 12:30:46 WARN Instrumentation: [369a3699] regParam is zero, which might cause numerical instability and overf

We have now tuned the elastic net model. Let's extract the cross-validated RMSE values and corresponding hyperparameter values for the ten models with the smallest RMSE. 

In [22]:
# Creating empty list to append model metric-hyperparameter pairs to
tune_list = []

# Appending model metric-hyperparameter pairs
for i in range(len(paramGrid)):
    tune_list.append([en_cv.avgMetrics[i], paramGrid[i].values()])

# Sorting the list by cross-validated log-loss
tune_list.sort(key = lambda x: x[0])

# Printing top 10 models
tune_list[0:10]

[[2174.9950671232627, dict_values([0.98, 0.5])],
 [2174.9954739810155, dict_values([0.05, 0.05])],
 [2174.9955552642587, dict_values([0.95, 0.5])],
 [2174.9957082453457, dict_values([0.25, 0.1])],
 [2174.995746516819, dict_values([0.25, 0.05])],
 [2174.9957466585734, dict_values([0.25, 0.25])],
 [2174.995747456857, dict_values([0.75, 0.75])],
 [2174.995808013895, dict_values([0.1, 0.5])],
 [2174.9958486438204, dict_values([0.1, 0.75])],
 [2174.995853207255, dict_values([0.1, 0.25])]]

These results are interesting: the top-performing model has a relatively large amount of regularization (`regParam` = 0.98) with regularization relatively balanced across the two penalty terms (`elasticNetParam` = 0.5). However, the second-best-performing model has very little total regularization (`regParam` = 0.05) and relies mostly on the ridge penalty (`elasticNetParam` = 0.05). This may be an indication that predicting `Power_Zone_3` is relatively robust to regularization scale and form. 

Overall, the cross-validated RMSE for each of the top models is roughly a third of `Power_Zone_3`'s standard deviation (6622.59), so our model is providing substantial benefit relative to simply using the sample mean to predict future power usage. 

To understand the quality of the training set fit, we can calculate the RMSE from predicting on our full dataset. This is as simple as using the `.evaluate()` method for `RegresssionEvaluator()` and passing `en_cv.transform(power_df)`. Our dataset will go through the pipeline and be "transformed" into predictions using the best-performing model fit to the same dataset. 

In [26]:
# Calculating full dataset RMSE
training_rmse = RegressionEvaluator().evaluate(en_cv.transform(power_df))
print(training_rmse)

2174.4502313329


As the model has seen all these observations before, it is not surprising that the training RMSE is a bit smaller than the cross-validated RMSE. 

To better understand all the work we've done up to this point, we can produce a final dataframe of true values, predictions, and residuals for the training set. We'll do this now and print the first few rows of data. 

In [31]:
# Creating table of true power values, predictions, and residuals
true_vs_predictions = en_cv.transform(power_df).select("label", "prediction").withColumn("residual", col("label") - col("prediction"))

# Extracting first five rows
true_vs_predictions.show(5)

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20931.634660509084|-690.6708005090841|
|20131.08434|18697.469312056288|1433.6150279437134|
|19668.43373|18233.227739866783|1435.2059901332177|
|18899.27711| 17612.13844916144|1287.1386608385583|
|18442.40964|17013.816095194572|1428.5935448054297|
+-----------+------------------+------------------+
only showing top 5 rows


Once the pipeline is set up, the MLlib workflow is truly streamlined. We easily obtained predictions appended to our processed dataframe and could easily create a table of true values and predictions for `Power_Zone_3`. 

## Streaming New Observations and Predicting Power Consumption

Now that we have a final fitted elastic net model, we can stream incoming observations and effectively put our model into production. This could be particularly helpful if we lost the ability to explicitly measure `Power_Zone_3` but still needed to know how much power was being consumed in that area. 

To showcase the ability to deploy our model on streamed data, we will simulate the streaming process by creating a `.py` file that randomly draws observations from a held-out set of observations and makes them visible when we begin streaming. Let's prepare for the stream by specifying the schema for the inbound data and setting up a `readStream`. Fortunately, the schema will match that of `power_df`. The data will come in via `.csv` files with headers and land in the `streamed_obs` folder in our local environment. We'll need to indicate all three facts for the `readStream`. 

In [ ]:
# Specifying our streaming data's input schema
myschema = power_df.schema

# Speciying the streaming data itself, which will arrive in streamed_obs folder
# Note that we need to specify that the data have headers
power_data = spark.readStream.schema(myschema) \
    .option("header", "true").csv("streamed_obs")

We are now ready to construct the separate transformations we will apply to the streamed `power_data` and indicate that we want to join these transformed dataframes on the true value of `Power_Zone_3` (denoted `label` after the transformations). 

The first transformation will effectively recreate `true_vs_predictions` above for the streamed observations, producing a true `label` value, `prediction`, and `residual`. The second transformation will simply rename `Power_Zone_3` `label` so we can merge the dataframes on `label`. 

A note on the join: we are performing an inner join, but the join type should not matter as we are merging transformations for the same inbound observations. 

In [ ]:
# Predicting Power_Zone_3 and producing residual
predictions = en_cv.transform(power_data).select("label", "prediction").withColumn("residual", col("label") - col("prediction"))

# Renaming Power_Zone_3 label in original streamed data
power_data_rename = power_data.withColumnRenamed("Power_Zone_3", "label")

# Merging transformed dataframes for streamed observations
joined = predictions \
                .join(power_data_rename, "label", "inner") 

Now that we've constructed the three objects we need to modify the streamed data, we simply need to start the stream. We'll indicate we want to append outputs as new data come in, and we'll also indicate we simply want the results printed to the console. 

In [ ]:
# Starting the stream to perform the two transformations and join results
# setting truncate to false to ensure we see all the resultant rows
joined.writeStream.outputMode("append") \
                .format("console") \
                .option("truncate", "false") \
                .start()